In [2]:
# ============================================================
# Stage 08-1. WordNet Data Augmentation Comparison
#
# 비교 실험:
#   1. Original Training Data
#   2. Original + WordNet Augmented Training Data
#
# Model: BERT-base-uncased
# Loss: Class-weighted Focal Loss
# Metric: Validation Macro F1
#
# 주요 수정:
#   - AMP에서 optimizer 업데이트가 실제로 수행된 경우에만
#     LR scheduler를 진행
#   - Gradient Accumulation의 마지막 불완전 그룹 처리
#   - 동일한 validation set 및 학습 설정으로 비교
#   - Test set은 사용하지 않음
# ============================================================

import os
import re
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

MODEL_NAME = "bert-base-uncased"
NUM_CLASSES = 6

TRAIN_PATH = Path("results/splits/train.csv")
VAL_PATH = Path("results/splits/validation.csv")

OUTPUT_DIR = Path("results/augmentation/wordnet_comparison")

# Training
EPOCHS = 3
MAX_LEN = 128

BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# Focal Loss
FOCAL_GAMMA = 2.0

# WordNet augmentation
AUGMENT_LABELS = {2, 3, 4, 5}
REPLACEMENT_PROBABILITY = 0.10
MAX_REPLACEMENTS_PER_TEXT = 2

# Reproducibility
NUM_WORKERS = 0

# CUDA AMP
USE_AMP = torch.cuda.is_available()

# Set True only if you intentionally want to regenerate augmentation.
REGENERATE_AUGMENTATION = True


# ============================================================
# 2. Random Seed
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Reproducibility can reduce performance.
    # Keep deterministic algorithms disabled for compatibility.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("Stage 08-1: WordNet Augmentation Comparison")
print("=" * 70)
print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {EPOCHS}")
print(f"Physical batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRAD_ACCUM_STEPS}")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max length: {MAX_LEN}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 3. Output Directories
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUGMENT_DIR = OUTPUT_DIR / "augmentation_data"
AUGMENT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_DIR = OUTPUT_DIR / "baseline"
AUGMENTED_DIR = OUTPUT_DIR / "wordnet_augmented"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. Load Data
# ============================================================

def load_dataset(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path.resolve()}"
        )

    df = pd.read_csv(path)

    required_columns = {"text", "label"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{path} must contain columns: "
            f"{sorted(required_columns)}"
        )

    df = df[["text", "label"]].copy()

    df["text"] = df["text"].fillna("").astype(str)
    df["label"] = pd.to_numeric(
        df["label"], errors="raise"
    ).astype(int)

    df = df[df["text"].str.strip().ne("")].reset_index(drop=True)

    invalid_labels = sorted(
        set(df["label"].unique()) - set(range(NUM_CLASSES))
    )

    if invalid_labels:
        raise ValueError(
            f"Invalid labels found: {invalid_labels}"
        )

    return df


train_df = load_dataset(TRAIN_PATH)
val_df = load_dataset(VAL_PATH)

print("\n[Dataset Information]")
print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")

print("\nTrain class counts:")
print(
    train_df["label"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nValidation class counts:")
print(
    val_df["label"]
    .value_counts()
    .sort_index()
    .to_string()
)

# The validation set must not be included in augmentation.
# The test set is intentionally never loaded in this script.


# ============================================================
# 5. Class Weights
# ============================================================

# Important:
# Calculate weights from the ORIGINAL training labels only.
# Use the exact same weights for both experiments.

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].to_numpy(),
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE,
)

print("\n[Class Weights]")
for label, weight in enumerate(class_weights_np):
    print(f"Class {label}: {weight:.4f}")


# ============================================================
# 6. WordNet Setup
# ============================================================

import nltk

from nltk.corpus import wordnet

def ensure_wordnet():
    """
    Download WordNet resources if they are not available.
    An internet connection may be required on the first run.
    """
    try:
        wordnet.synsets("happy")
    except LookupError:
        print("\nDownloading NLTK WordNet resources...")
        nltk.download("wordnet", quiet=False)

        try:
            wordnet.synsets("happy")
        except LookupError as exc:
            raise RuntimeError(
                "WordNet resources could not be loaded. "
                "Check your internet connection and NLTK installation."
            ) from exc


ensure_wordnet()


# ============================================================
# 7. WordNet Augmentation
# ============================================================

# These lists reduce some obvious semantic distortions.
# They cannot guarantee that every replacement preserves meaning.

PROTECTED_WORDS = {
    # Negation and uncertainty
    "no", "not", "never", "none", "nothing", "nobody",
    "neither", "nor", "without", "cannot", "cant",
    "don't", "dont", "doesn't", "doesnt",
    "didn't", "didnt", "isn't", "isnt",
    "wasn't", "wasnt", "won't", "wont",
    "wouldn't", "wouldnt", "couldn't", "couldnt",
    "shouldn't", "shouldnt",

    # Pronouns and common function words
    "i", "me", "my", "mine", "we", "us", "our",
    "you", "your", "yours", "he", "him", "his",
    "she", "her", "hers", "it", "its", "they",
    "them", "their", "theirs", "this", "that",
    "these", "those", "a", "an", "the",
    "and", "or", "but", "if", "because", "so",
    "to", "of", "for", "with", "from", "in", "on",
    "at", "by", "as", "than", "then", "when",
    "where", "while", "what", "which", "who",
    "whom", "whose", "how", "why",

    # Auxiliary verbs and contractions
    "am", "is", "are", "was", "were", "be", "been",
    "being", "do", "does", "did", "have", "has",
    "had", "can", "could", "will", "would", "shall",
    "should", "may", "might", "must",

    # Intensifiers and frequency words
    "very", "really", "extremely", "too", "quite",
    "so", "just", "almost", "barely", "hardly",
    "always", "never", "often", "sometimes",
}

# Protect common emotion words to reduce direct changes
# to the emotion-bearing vocabulary.
EMOTION_WORDS = {
    "happy", "happiness", "joy", "joyful", "excited",
    "love", "loved", "loving", "like", "liked",
    "sad", "sadness", "unhappy", "depressed",
    "angry", "anger", "mad", "furious", "annoyed",
    "fear", "afraid", "scared", "terrified", "anxious",
    "surprised", "surprise", "shocked", "disgusted",
    "disgust", "hate", "hated", "hating",
    "worried", "worry", "hope", "hopeful",
    "lonely", "nervous", "upset", "calm",
}


# Match an alphabetic word while preserving surrounding punctuation.
WORD_PATTERN = re.compile(
    r"^([^A-Za-z]*)([A-Za-z]+(?:'[A-Za-z]+)?)([^A-Za-z]*)$"
)


def preserve_case(original, replacement):
    """Preserve the original token's capitalization pattern."""
    if original.isupper():
        return replacement.upper()

    if original.istitle():
        return replacement.capitalize()

    return replacement.lower()


def get_wordnet_synonyms(word):
    """
    Get simple WordNet synonyms.

    Limitation:
    WordNet synonyms are not necessarily interchangeable in context.
    The function filters candidates, but cannot fully understand sentence
    meaning, sentiment, grammar, or emotion-label consistency.
    """
    word_lower = word.lower()

    if word_lower in PROTECTED_WORDS:
        return []

    if word_lower in EMOTION_WORDS:
        return []

    candidates = set()

    for synset in wordnet.synsets(word_lower):
        for lemma in synset.lemmas():
            candidate = lemma.name().lower()

            # Avoid multiword expressions and malformed candidates.
            if "_" in candidate or " " in candidate:
                continue

            if not candidate.isalpha():
                continue

            if candidate == word_lower:
                continue

            if len(candidate) < 3:
                continue

            if candidate in PROTECTED_WORDS:
                continue

            if candidate in EMOTION_WORDS:
                continue

            candidates.add(candidate)

    return sorted(candidates)


def augment_text(
    text,
    replacement_probability=REPLACEMENT_PROBABILITY,
    max_replacements=MAX_REPLACEMENTS_PER_TEXT,
):
    """
    Replace a limited number of words with WordNet synonyms.

    Returns:
        augmented_text, number_of_replacements
    """
    if not isinstance(text, str) or not text.strip():
        return text, 0

    tokens = text.split()
    eligible_positions = []

    for i, token in enumerate(tokens):
        match = WORD_PATTERN.match(token)

        if not match:
            continue

        prefix, word, suffix = match.groups()

        if word.lower() in PROTECTED_WORDS:
            continue

        if word.lower() in EMOTION_WORDS:
            continue

        if len(word) < 3:
            continue

        synonyms = get_wordnet_synonyms(word)

        if synonyms:
            eligible_positions.append(
                (i, prefix, word, suffix, synonyms)
            )

    if not eligible_positions:
        return text, 0

    random.shuffle(eligible_positions)

    replaced_count = 0

    for (
        position,
        prefix,
        original_word,
        suffix,
        synonyms,
    ) in eligible_positions:

        if replaced_count >= max_replacements:
            break

        if random.random() > replacement_probability:
            continue

        replacement = random.choice(synonyms)
        replacement = preserve_case(
            original_word,
            replacement,
        )

        tokens[position] = (
            prefix + replacement + suffix
        )

        replaced_count += 1

    return " ".join(tokens), replaced_count


def create_wordnet_dataset(train_data):
    """
    Create at most one augmented version per eligible original row.

    Original rows are retained. Augmented rows are appended to them.
    Only the original training set is used to generate new examples.
    """
    set_seed(SEED)

    augmented_records = []
    class_summary = []

    original_counts = (
        train_data["label"]
        .value_counts()
        .reindex(range(NUM_CLASSES), fill_value=0)
    )

    for label in range(NUM_CLASSES):
        class_rows = train_data[
            train_data["label"] == label
        ]

        if label not in AUGMENT_LABELS:
            class_summary.append({
                "label": label,
                "original_count": int(original_counts[label]),
                "augmented_count": 0,
                "augmentation_rate_pct": 0.0,
            })
            continue

        class_augmented_count = 0

        print(
            f"\nGenerating augmentation for class {label} "
            f"({len(class_rows):,} rows)..."
        )

        for row in class_rows.itertuples(index=False):
            original_text = str(row.text)

            augmented_text, replacements = augment_text(
                original_text
            )

            # Do not append unchanged examples.
            if (
                replacements > 0
                and augmented_text.strip()
                and augmented_text != original_text
            ):
                augmented_records.append({
                    "text": augmented_text,
                    "label": int(row.label),
                    "source": "wordnet_augmented",
                    "replacements": replacements,
                })

                class_augmented_count += 1

        rate = (
            class_augmented_count / len(class_rows) * 100
            if len(class_rows) > 0
            else 0.0
        )

        class_summary.append({
            "label": label,
            "original_count": int(original_counts[label]),
            "augmented_count": class_augmented_count,
            "augmentation_rate_pct": rate,
        })

        print(
            f"Class {label}: "
            f"{class_augmented_count:,} augmented rows "
            f"({rate:.2f}%)"
        )

    augmented_examples = pd.DataFrame(
        augmented_records,
        columns=[
            "text",
            "label",
            "source",
            "replacements",
        ],
    )

    summary_df = pd.DataFrame(class_summary)

    return augmented_examples, summary_df


# ============================================================
# 8. Generate or Load Augmented Data
# ============================================================

AUGMENTED_EXAMPLES_PATH = (
    AUGMENT_DIR / "augmented_examples.csv"
)

AUGMENTED_TRAIN_PATH = (
    AUGMENT_DIR / "augmented_train.csv"
)

CLASS_SUMMARY_PATH = (
    AUGMENT_DIR / "augmentation_class_summary.csv"
)

if (
    REGENERATE_AUGMENTATION
    or not AUGMENTED_EXAMPLES_PATH.exists()
    or not AUGMENTED_TRAIN_PATH.exists()
    or not CLASS_SUMMARY_PATH.exists()
):
    print("\n[WordNet Augmentation] Generating data...")

    augmented_examples, augmentation_summary = (
        create_wordnet_dataset(train_df)
    )

    augmented_examples.to_csv(
        AUGMENTED_EXAMPLES_PATH,
        index=False,
    )

    # Training data used by the augmented experiment:
    # original examples + newly generated examples.
    augmented_train_df = pd.concat(
        [
            train_df[["text", "label"]].copy(),
            augmented_examples[["text", "label"]].copy(),
        ],
        ignore_index=True,
    )

    augmented_train_df.to_csv(
        AUGMENTED_TRAIN_PATH,
        index=False,
    )

    augmentation_summary.to_csv(
        CLASS_SUMMARY_PATH,
        index=False,
    )

else:
    print("\n[WordNet Augmentation] Loading saved data...")

    augmented_examples = pd.read_csv(
        AUGMENTED_EXAMPLES_PATH
    )

    augmented_train_df = load_dataset(
        AUGMENTED_TRAIN_PATH
    )

    augmentation_summary = pd.read_csv(
        CLASS_SUMMARY_PATH
    )


print("\n[Augmentation Summary]")
print(augmentation_summary.to_string(index=False))

print("\n[Dataset Sizes]")
print(f"Original train: {len(train_df):,}")
print(f"Generated examples: {len(augmented_examples):,}")
print(f"Augmented train: {len(augmented_train_df):,}")

if len(train_df) > 0:
    increase_pct = (
        len(augmented_train_df) - len(train_df)
    ) / len(train_df) * 100

    print(f"Training data increase: {increase_pct:.2f}%")

print("\n[Augmentation Examples]")
if len(augmented_examples) > 0:
    print(
        augmented_examples[
            ["label", "text", "replacements"]
        ]
        .head(20)
        .to_string(index=False)
    )
else:
    print("No examples were generated.")


# ============================================================
# 9. Tokenizer
# ============================================================

print("\n[Tokenizer] Loading tokenizer...")

tokenizer = BertTokenizerFast.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 10. Dataset and Collate Function
# ============================================================

class EmotionDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx],
        }


def make_collate_fn(tokenizer, max_length):
    def collate_fn(batch):
        texts = [item["text"] for item in batch]

        labels = torch.tensor(
            [item["label"] for item in batch],
            dtype=torch.long,
        )

        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded["labels"] = labels

        return encoded

    return collate_fn


def make_dataloader(dataframe, shuffle):
    dataset = EmotionDataset(dataframe)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=make_collate_fn(
            tokenizer,
            MAX_LEN,
        ),
    )


# ============================================================
# 11. Class-weighted Focal Loss
# ============================================================

class WeightedFocalLoss(nn.Module):
    """
    Focal Loss with class weights.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Reduction:
        sum(weighted focal losses) / sum(class weights in batch)

    The same fixed class weights are used in both experiments.
    """

    def __init__(self, class_weights, gamma=2.0):
        super().__init__()

        self.register_buffer(
            "class_weights",
            class_weights.detach().clone(),
        )

        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = torch.nn.functional.log_softmax(
            logits,
            dim=-1,
        )

        log_pt = log_probs.gather(
            dim=1,
            index=targets.unsqueeze(1),
        ).squeeze(1)

        pt = log_pt.exp()

        alpha_t = self.class_weights[targets]

        focal_factor = (1.0 - pt).pow(self.gamma)

        per_example_loss = (
            -alpha_t * focal_factor * log_pt
        )

        denominator = alpha_t.sum().clamp_min(1e-12)

        return per_example_loss.sum() / denominator


# ============================================================
# 12. Evaluation
# ============================================================

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_labels = []
    all_predictions = []

    for batch in dataloader:
        labels = batch.pop("labels").to(DEVICE)

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            outputs = model(**batch)
            logits = outputs.logits

            loss = criterion(logits, labels)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_examples += batch_size

        predictions = torch.argmax(
            logits,
            dim=-1,
        )

        all_labels.extend(
            labels.detach().cpu().numpy().tolist()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy().tolist()
        )

    average_loss = (
        total_loss / total_examples
        if total_examples > 0
        else 0.0
    )

    metrics = {
        "val_loss": average_loss,
        "val_accuracy": accuracy_score(
            all_labels,
            all_predictions,
        ),
        "val_macro_precision": precision_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0,
        ),
        "val_macro_recall": recall_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0,
        ),
        "val_macro_f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0,
        ),
        "val_weighted_f1": f1_score(
            all_labels,
            all_predictions,
            average="weighted",
            zero_division=0,
        ),
    }

    return metrics, all_labels, all_predictions


# ============================================================
# 13. Save Validation Results
# ============================================================

def save_validation_outputs(
    run_dir,
    labels,
    predictions,
):
    report = classification_report(
        labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose()

    report_df.to_csv(
        run_dir / "validation_classification_report.csv"
    )

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
    )

    matrix_df = pd.DataFrame(
        matrix,
        index=[
            f"actual_{i}" for i in range(NUM_CLASSES)
        ],
        columns=[
            f"predicted_{i}" for i in range(NUM_CLASSES)
        ],
    )

    matrix_df.to_csv(
        run_dir / "validation_confusion_matrix.csv"
    )


# ============================================================
# 14. Training
# ============================================================

def train_experiment(
    experiment_name,
    training_df,
    run_dir,
):
    """
    Train one experiment and select the best checkpoint
    using validation Macro F1.

    AMP scheduler fix:
        - If GradScaler decreases its scale, the optimizer
          update was skipped due to non-finite gradients.
        - Do not advance the scheduler in that case.
    """
    print("\n" + "=" * 70)
    print(f"Experiment: {experiment_name}")
    print("=" * 70)

    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = make_dataloader(
        training_df,
        shuffle=True,
    )

    val_loader = make_dataloader(
        val_df,
        shuffle=False,
    )

    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
    ).to(DEVICE)

    criterion = WeightedFocalLoss(
        class_weights=class_weights,
        gamma=FOCAL_GAMMA,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    # Number of optimizer updates per epoch.
    updates_per_epoch = (
        len(train_loader) + GRAD_ACCUM_STEPS - 1
    ) // GRAD_ACCUM_STEPS

    total_training_steps = updates_per_epoch * EPOCHS

    warmup_steps = int(
        total_training_steps * WARMUP_RATIO
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

    # Use the modern torch.amp API.
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    history = []

    best_val_macro_f1 = -1.0
    best_epoch = 0

    best_checkpoint_path = (
        run_dir / "best_model.safetensors"
    )

    best_predictions = None
    best_labels = None

    optimizer.zero_grad(set_to_none=True)

    total_start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()

        epoch_start_time = time.time()

        running_loss = 0.0
        total_examples = 0

        optimizer.zero_grad(set_to_none=True)

        num_batches = len(train_loader)

        for batch_idx, batch in enumerate(train_loader):
            labels = batch.pop("labels").to(DEVICE)

            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            # Handle a final accumulation group that contains
            # fewer batches than GRAD_ACCUM_STEPS.
            group_start = (
                batch_idx // GRAD_ACCUM_STEPS
            ) * GRAD_ACCUM_STEPS

            group_size = min(
                GRAD_ACCUM_STEPS,
                num_batches - group_start,
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                outputs = model(**batch)

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels,
                )

                # Divide by the actual group size, including
                # the final partial accumulation group.
                scaled_loss = loss / group_size

            scaler.scale(scaled_loss).backward()

            batch_size = labels.size(0)

            running_loss += loss.detach().item() * batch_size
            total_examples += batch_size

            is_accumulation_boundary = (
                (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
            )

            is_last_batch = (
                batch_idx + 1 == num_batches
            )

            if (
                is_accumulation_boundary
                or is_last_batch
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=MAX_GRAD_NORM,
                )

                old_scale = scaler.get_scale()

                # GradScaler may skip this update if gradients
                # contain non-finite values.
                scaler.step(optimizer)
                scaler.update()

                new_scale = scaler.get_scale()

                # If the scale decreased, GradScaler skipped
                # optimizer.step(). Do not advance the scheduler.
                #
                # If the scale is unchanged or increases,
                # the optimizer update was performed.
                if new_scale >= old_scale:
                    scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        train_loss = (
            running_loss / total_examples
            if total_examples > 0
            else 0.0
        )

        # Validation
        val_metrics, val_labels, val_predictions = evaluate(
            model,
            val_loader,
            criterion,
        )

        epoch_time = (
            time.time() - epoch_start_time
        ) / 60.0

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics,
            "epoch_time_min": epoch_time,
            "learning_rate_next": scheduler.get_last_lr()[0],
        }

        history.append(record)

        print(
            f"\nEpoch {epoch}/{EPOCHS}"
        )

        print(
            f"Train Loss: {train_loss:.6f}"
        )

        print(
            f"Validation Loss: "
            f"{val_metrics['val_loss']:.6f}"
        )

        print(
            f"Validation Accuracy: "
            f"{val_metrics['val_accuracy']:.6f}"
        )

        print(
            f"Validation Macro F1: "
            f"{val_metrics['val_macro_f1']:.6f}"
        )

        print(
            f"Validation Weighted F1: "
            f"{val_metrics['val_weighted_f1']:.6f}"
        )

        print(
            f"Epoch time: {epoch_time:.2f} min"
        )

        # Select the checkpoint using validation Macro F1.
        current_macro_f1 = val_metrics["val_macro_f1"]

        if current_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = current_macro_f1
            best_epoch = epoch

            best_labels = list(val_labels)
            best_predictions = list(val_predictions)

            # Save only the best checkpoint for this experiment.
            # safetensors avoids pickle-based model weight loading.
            from safetensors.torch import save_file

            state_dict_cpu = {
                key: value.detach().cpu().contiguous()
                for key, value in model.state_dict().items()
            }

            save_file(
                state_dict_cpu,
                str(best_checkpoint_path),
            )

            print(
                f"Saved best checkpoint: "
                f"{best_checkpoint_path}"
            )

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    total_time_min = (
        time.time() - total_start_time
    ) / 60.0

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        run_dir / "training_history.csv",
        index=False,
    )

    save_validation_outputs(
        run_dir,
        best_labels,
        best_predictions,
    )

    best_record = history_df.loc[
        history_df["val_macro_f1"].idxmax()
    ].to_dict()

    result = {
        "experiment": experiment_name,
        "train_samples": int(len(training_df)),
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "best_val_accuracy": float(
            best_record["val_accuracy"]
        ),
        "best_val_macro_precision": float(
            best_record["val_macro_precision"]
        ),
        "best_val_macro_recall": float(
            best_record["val_macro_recall"]
        ),
        "best_val_weighted_f1": float(
            best_record["val_weighted_f1"]
        ),
        "best_val_loss": float(
            best_record["val_loss"]
        ),
        "total_training_time_min": float(
            total_time_min
        ),
        "mean_epoch_time_min": float(
            history_df["epoch_time_min"].mean()
        ),
        "best_checkpoint": str(
            best_checkpoint_path
        ),
    }

    with open(
        run_dir / "experiment_summary.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=4,
        )

    print("\n[Best Experiment Result]")
    for key, value in result.items():
        print(f"{key}: {value}")

    del model
    del optimizer
    del scheduler
    del scaler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


# ============================================================
# 15. Run Baseline and Augmentation Experiments
# ============================================================

# Both experiments use:
# - The same original validation set
# - The same initial pretrained checkpoint
# - The same optimizer and loss configuration
# - The same class weights computed from original train labels
# - The same number of epochs
#
# Training data differs only in whether augmented examples
# are appended to the original training data.

baseline_result = train_experiment(
    experiment_name="Original Training Data",
    training_df=train_df,
    run_dir=BASELINE_DIR,
)

augmented_result = train_experiment(
    experiment_name="Original + WordNet Augmentation",
    training_df=augmented_train_df,
    run_dir=AUGMENTED_DIR,
)


# ============================================================
# 16. Compare Results
# ============================================================

comparison_df = pd.DataFrame(
    [
        baseline_result,
        augmented_result,
    ]
)

comparison_path = (
    OUTPUT_DIR / "comparison_summary.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

print("\n" + "=" * 70)
print("Final Comparison")
print("=" * 70)

display_columns = [
    "experiment",
    "train_samples",
    "best_epoch",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_macro_precision",
    "best_val_macro_recall",
    "best_val_weighted_f1",
    "total_training_time_min",
]

print(
    comparison_df[display_columns].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 17. Calculate Differences
# ============================================================

baseline_f1 = baseline_result["best_val_macro_f1"]
augmented_f1 = augmented_result["best_val_macro_f1"]

f1_difference = augmented_f1 - baseline_f1

baseline_time = baseline_result["total_training_time_min"]
augmented_time = augmented_result["total_training_time_min"]

if baseline_time > 0:
    time_increase_pct = (
        (augmented_time - baseline_time)
        / baseline_time
        * 100
    )
else:
    time_increase_pct = float("nan")

print("\n[Difference: Augmented - Baseline]")

print(
    f"Validation Macro F1 difference: "
    f"{f1_difference:+.6f}"
)

print(
    f"Validation Macro F1 difference (percentage points): "
    f"{f1_difference * 100:+.4f}%p"
)

print(
    f"Training time increase: "
    f"{time_increase_pct:+.2f}%"
)


if f1_difference > 0:
    print(
        "\nThe augmentation experiment achieved a higher "
        "best validation Macro F1 in this run."
    )
elif f1_difference < 0:
    print(
        "\nThe baseline achieved a higher best validation "
        "Macro F1 in this run."
    )
else:
    print(
        "\nBoth experiments achieved the same best "
        "validation Macro F1 in this run."
    )

print(
    "\nImportant: A single run does not establish statistical "
    "significance. Inspect the generated examples and per-class "
    "metrics before deciding whether to use augmentation."
)

print("\n[Saved Results]")
print(f"Comparison: {comparison_path.resolve()}")
print(f"Augmentation examples: {AUGMENTED_EXAMPLES_PATH.resolve()}")
print(f"Augmentation summary: {CLASS_SUMMARY_PATH.resolve()}")
print(f"Baseline results: {BASELINE_DIR.resolve()}")
print(f"Augmented results: {AUGMENTED_DIR.resolve()}")

print("\nStage 08-1 completed.")

Stage 08-1: WordNet Augmentation Comparison
Device: cuda
AMP enabled: True
Model: bert-base-uncased
Epochs: 3
Physical batch size: 16
Gradient accumulation: 4
Effective batch size: 64
Learning rate: 2e-05
Max length: 128
Output directory: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/augmentation/wordnet_comparison
GPU: NVIDIA GeForce RTX 5070 Ti

[Dataset Information]
Train rows: 305,345
Validation rows: 65,422

Train class counts:
label
0     88764
1    103242
2     25336
3     41972
4     35041
5     10990

Validation class counts:
label
0    18986
1    22211
2     5377
3     9013
4     7505
5     2330

[Class Weights]
Class 0: 0.5733
Class 1: 0.4929
Class 2: 2.0086
Class 3: 1.2125
Class 4: 1.4523
Class 5: 4.6306

[WordNet Augmentation] Generating data...

Generating augmentation for class 2 (25,336 rows)...
Class 2: 13,754 augmented rows (54.29%)

Generating augmentation for class 3 (41,972 rows)...
Class 3: 21,601 augmented rows (51.47%)

Generating augme

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Experiment: Original Training Data


...

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/3
Train Loss: 0.163344
Validation Loss: 0.030229
Validation Accuracy: 0.942466
Validation Macro F1: 0.917646
Validation Weighted F1: 0.944317
Epoch time: 8.35 min
Saved best checkpoint: results/augmentation/wordnet_comparison/baseline/best_model.safetensors

Epoch 2/3
Train Loss: 0.029362
Validation Loss: 0.029337
Validation Accuracy: 0.942542
Validation Macro F1: 0.917638
Validation Weighted F1: 0.944418
Epoch time: 8.41 min

Epoch 3/3
Train Loss: 0.026036
Validation Loss: 0.029880
Validation Accuracy: 0.942970
Validation Macro F1: 0.918226
Validation Weighted F1: 0.944823
Epoch time: 8.80 min
Saved best checkpoint: results/augmentation/wordnet_comparison/baseline/best_model.safetensors

[Best Experiment Result]
experiment: Original Training Data
train_samples: 305345
best_epoch: 3
best_val_macro_f1: 0.9182259416573123
best_val_accuracy: 0.9429702546543977
best_val_macro_precision: 0.892081537251404
best_val_macro_recall: 0.9565990473413732
best_val_weighted_f1: 0.94482288957

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/3
Train Loss: 0.167418
Validation Loss: 0.032435
Validation Accuracy: 0.940341
Validation Macro F1: 0.915287
Validation Weighted F1: 0.942219
Epoch time: 10.17 min
Saved best checkpoint: results/augmentation/wordnet_comparison/wordnet_augmented/best_model.safetensors

Epoch 2/3
Train Loss: 0.033635
Validation Loss: 0.030529
Validation Accuracy: 0.942145
Validation Macro F1: 0.917325
Validation Weighted F1: 0.944020
Epoch time: 10.35 min
Saved best checkpoint: results/augmentation/wordnet_comparison/wordnet_augmented/best_model.safetensors

Epoch 3/3
Train Loss: 0.027156
Validation Loss: 0.030644
Validation Accuracy: 0.942313
Validation Macro F1: 0.917496
Validation Weighted F1: 0.944200
Epoch time: 10.15 min
Saved best checkpoint: results/augmentation/wordnet_comparison/wordnet_augmented/best_model.safetensors

[Best Experiment Result]
experiment: Original + WordNet Augmentation
train_samples: 364874
best_epoch: 3
best_val_macro_f1: 0.9174957654482024
best_val_accuracy: 0.9423